In [14]:
import pandas as pd
import re

# loading the merchants table
merchants = pd.read_parquet('../data/tables/tbl_merchants.parquet')
print(merchants.shape)
print(merchants.dtypes)
merchants.head()


(4026, 2)
name    object
tags    object
dtype: object


,name,tags
merchant_abn,,
10023283211,Felis Limited,"((furniture, home furnishings and equipment sh..."
10142254217,Arcu Ac Orci Corporation,"([cable, satellite, and otHer pay television a..."
10165489824,Nunc Sed Company,"([jewelry, watch, clock, and silverware shops]..."
10187291046,Ultricies Dignissim Lacus Foundation,"([wAtch, clock, and jewelry repair shops], [b]..."
10192359162,Enim Condimentum PC,"([music shops - musical instruments, pianos, a..."


In [15]:
# print out the tags to understand the format
for t in merchants['tags'].head(5):
    print(t)
    print('\n')

((furniture, home furnishings and equipment shops, and manufacturers, except appliances), (e), (take rate: 0.18))


([cable, satellite, and otHer pay television and radio services], [b], [take rate: 4.22])


([jewelry, watch, clock, and silverware shops], [b], [take rate: 4.40])


([wAtch, clock, and jewelry repair shops], [b], [take rate: 3.29])


([music shops - musical instruments, pianos, and sheet music], [a], [take rate: 6.33])




In [16]:
# divide tag into 3 columns of category, revenue_band, take_rate
def parse_tags(tag_str):
    m = re.search(
        r'[\(\[](.+?)[\)\]],\s*[\(\[]([a-eA-E])[\)\]],\s*[\(\[]take rate:\s*([\d.]+)[\)\]]',
        tag_str, flags=re.IGNORECASE
    )
    if not m:
        return pd.Series({'category': None, 'revenue_band': None, 'take_rate': None})
    
    category_raw = m.group(1)
    # remove any leading or trailing brackets or parentheses from the category
    category_clean = re.sub(r'^[\[\(]+|[\]\)]+$', '', category_raw).strip()
    
    return pd.Series({
        'category': category_clean,
        'revenue_band': m.group(2).lower(),
        'take_rate': float(m.group(3))
    })

parsed = merchants['tags'].apply(parse_tags)
merchants = pd.concat([merchants, parsed], axis=1)
merchants[['name', 'category', 'revenue_band', 'take_rate']].head(10)

,name,category,revenue_band,take_rate
merchant_abn,,,,
10023283211,Felis Limited,"furniture, home furnishings and equipment shop...",e,0.18
10142254217,Arcu Ac Orci Corporation,"cable, satellite, and otHer pay television and...",b,4.22
10165489824,Nunc Sed Company,"jewelry, watch, clock, and silverware shops",b,4.40
10187291046,Ultricies Dignissim Lacus Foundation,"wAtch, clock, and jewelry repair shops",b,3.29
10192359162,Enim Condimentum PC,"music shops - musical instruments, pianos, and...",a,6.33
10206519221,Fusce Company,"gift, card, novelty, and souvenir shops",a,6.34
10255988167,Aliquam Enim Incorporated,"computers, comPUter peripheral equipment, and ...",b,4.32
10264435225,Ipsum Primis Ltd,"watch, clock, and jewelry repair shops",c,2.39
10279061213,Pede Ultrices Industries,"computer programming , data processing, and in...",a,5.71


In [17]:
# checking any null values, duplicates, and the range of take_rate and revenue_band values
print('Null counts:')
print(merchants[['category', 'revenue_band', 'take_rate']].isnull().sum())

print('\nDuplicate merchant_abn:', merchants.index.duplicated().sum())

print('\nTake rate range:', merchants['take_rate'].min(), '-', merchants['take_rate'].max())
print('Revenue band values:', sorted(merchants['revenue_band'].dropna().unique()))

Null counts:
category        0
revenue_band    0
take_rate       0
dtype: int64

Duplicate merchant_abn: 0

Take rate range: 0.1 - 7.0
Revenue band values: ['a', 'b', 'c', 'd', 'e']


In [18]:
# Load all transaction parquet from 3 folder snapshot (2021-2022)
import glob

txn_files = glob.glob('../data/tables/transactions_*_snapshot/order_datetime=*/*.parquet')
print(f'Number of file transaction: {len(txn_files)}')

txn_list = [pd.read_parquet(f) for f in txn_files]
transactions = pd.concat(txn_list, ignore_index=True)
print(transactions.shape)
transactions.head()

Number of file transaction: 606
(14195505, 4)


,user_id,merchant_abn,dollar_value,order_id
0,18482,89295426212,132.460994,5cea6247-6cac-42c4-95a6-9a9a736dea67
1,1,74019238521,5.228331,9138e9c7-3c1a-40da-83dc-8bbe3e14e2cc
2,18484,45629217853,0.135488,5670e543-c3f6-4d59-aa4b-cfd2ec5d6130
3,2,64203420245,30.681990,690d590e-caa3-489a-a73d-42d7786c54f7
4,18484,96513857365,231.252946,c838a359-60f4-40f5-a274-787d03a5469f


In [19]:
# calculate the total revenue, number of transactions, and average order value for each merchant
merchant_txn_agg = transactions.groupby('merchant_abn').agg(
    total_revenue=('dollar_value', 'sum'),
    n_transactions=('dollar_value', 'count'),
    avg_order_value=('dollar_value', 'mean')
).reset_index()

print(merchant_txn_agg.shape)
merchant_txn_agg.head()

(4422, 4)


,merchant_abn,total_revenue,n_transactions,avg_order_value
0,10023283211,703277.711451,3261,215.663205
1,10142254217,118356.146073,3036,38.984238
2,10165489824,56180.473857,5,11236.094771
3,10187291046,39693.730387,336,118.136102
4,10192359162,177980.505456,385,462.287027


In [20]:
# Load merchant fraud probability, merge with the merchant table
merchant_fraud = pd.read_csv('../data/tables/merchant_fraud_probability.csv')
print(merchant_fraud.shape)
merchant_fraud.head()

(114, 3)


,merchant_abn,order_datetime,fraud_probability
0,19492220327,2021-11-28,44.403659
1,31334588839,2021-10-02,42.755301
2,19492220327,2021-12-22,38.867790
3,82999039227,2021-12-19,94.134700
4,90918180829,2021-09-02,43.325517


In [21]:
# merchant_fraud with merchant on date and take average fraud_probabilty of each merchant
merchant_fraud_agg = merchant_fraud.groupby('merchant_abn').agg(
    avg_fraud_probability=('fraud_probability', 'mean')
).reset_index()

print(merchant_fraud_agg.shape)
merchant_fraud_agg.head()

(61, 2)


,merchant_abn,avg_fraud_probability
0,11149063370,53.286933
1,11470993597,63.377344
2,11590404675,29.607818
3,14530561097,80.800545
4,14827550074,42.000567


In [22]:
# reset the index of merchants to merge with the aggregated transaction and fraud data
merchant_final = merchants.reset_index().merge(
    merchant_txn_agg, on='merchant_abn', how='left'
).merge(
    merchant_fraud_agg, on='merchant_abn', how='left'
)

print(merchant_final.shape)
merchant_final.head()

(4026, 10)


,merchant_abn,name,tags,category,revenue_band,take_rate,total_revenue,n_transactions,avg_order_value,avg_fraud_probability
0,10023283211,Felis Limited,"((furniture, home furnishings and equipment sh...","furniture, home furnishings and equipment shop...",e,0.18,703277.711451,3261,215.663205,NaN
1,10142254217,Arcu Ac Orci Corporation,"([cable, satellite, and otHer pay television a...","cable, satellite, and otHer pay television and...",b,4.22,118356.146073,3036,38.984238,NaN
2,10165489824,Nunc Sed Company,"([jewelry, watch, clock, and silverware shops]...","jewelry, watch, clock, and silverware shops",b,4.40,56180.473857,5,11236.094771,NaN
3,10187291046,Ultricies Dignissim Lacus Foundation,"([wAtch, clock, and jewelry repair shops], [b]...","wAtch, clock, and jewelry repair shops",b,3.29,39693.730387,336,118.136102,NaN
4,10192359162,Enim Condimentum PC,"([music shops - musical instruments, pianos, a...","music shops - musical instruments, pianos, and...",a,6.33,177980.505456,385,462.287027,NaN


In [24]:
# filter merchants with at least 1 transaction 
# (remove merchants with no transactions - n_transactions will be NaN)
merchant_clean = merchant_final[
    (merchant_final['n_transactions'].notna()) &
    (merchant_final['n_transactions'] >= 1) &
    (merchant_final['take_rate'] > 0)
].copy()

print(f'Before filtering: {len(merchant_final)} merchant')
print(f'After filtering: {len(merchant_clean)} merchant')

# save the cleaned merchant table to a parquet file
merchant_clean.to_parquet('../data/tables/merchant_clean.parquet', index=False)


Before filtering: 4026 merchant
After filtering: 4026 merchant
